# Test Scores Analysis
This notebook analyzes test scores from `test_scores.csv`. The metrics in the CSV are stored as string tuples formatted like `(score, weight)`, so this notebook parses them to calculate accurate weighted averages.

In [24]:
import pandas as pd
import ast

# Path to the data (relative to the eval/ folder)
file_path = "../exp_output/nova3r_points_cond_finetune_complete/nova3r_training/version_37/test_scores.csv"

# Load the data
df = pd.read_csv(file_path)
display(df.head())

,CD_16384_full,f_score_0.1_full,f_score_0.05_full,f_score_0.02_full,seq_name,id
0,"(0.01777886226773262, 10)","(0.9947611093521118, 10)","(0.9732902646064758, 10)","(0.7176934480667114, 10)",replica_pano_large_apartment_2_003,0
1,"(0.014563784003257751, 10)","(0.9961110949516296, 10)","(0.979584813117981, 10)","(0.8225622177124023, 10)",replica_pano_hotel_0_000,0
2,"(0.017383238300681114, 10)","(0.9946767091751099, 10)","(0.9654464721679688, 10)","(0.7441661953926086, 10)",replica_pano_large_apartment_2_002,0
3,"(0.03270377963781357, 10)","(0.9502124786376953, 10)","(0.8810319900512695, 10)","(0.4387843608856201, 10)",replica_pano_large_apartment_2_001,1
4,"(0.017430663108825684, 10)","(0.9948936700820923, 10)","(0.9723787307739258, 10)","(0.7603546380996704, 10)",replica_pano_large_apartment_0_004,1


In [25]:
# Identify the metric columns (everything except seq_name and id)
metric_cols = [col for col in df.columns if col not in ['seq_name', 'id']]

def parse_metric(val_str):
    """Parses a string like '(0.07, 1)' into a Python tuple."""
    if pd.isna(val_str):
        return 0.0, 0
    if isinstance(val_str, str):
        try:
            return ast.literal_eval(val_str)
        except Exception:
            return 0.0, 0
    return 0.0, 0

def get_weighted_average(series):
    """Calculates the weighted average for a pandas Series containing tuple strings."""
    parsed = series.apply(parse_metric)
    scores = parsed.apply(lambda x: x[0])
    weights = parsed.apply(lambda x: x[1])
    
    total_weight = weights.sum()
    if total_weight > 0:
        return (scores * weights).sum() / total_weight
    return 0.0
    
def get_total_weight(series):
    """Calculates the sum of weights for a pandas Series containing tuple strings."""
    parsed = series.apply(parse_metric)
    weights = parsed.apply(lambda x: x[1])
    return weights.sum()

## Total Metrics
Calculations for the total combined weighted averages across all sequences.

In [26]:
print("--- Total Weighted Averages ---")
total_metrics = {}
for col in metric_cols:
    total_metrics[col] = get_weighted_average(df[col])
    print(f"{col}: {total_metrics[col]:.6f}")

--- Total Weighted Averages ---
CD_16384_full: 0.019972
f_score_0.1_full: 0.986131
f_score_0.05_full: 0.954346
f_score_0.02_full: 0.696712


## Per-Scene Metrics
Calculations for average metrics individually grouped by `seq_name`.

In [27]:
scene_results = []

for scene_name, group in df.groupby('seq_name'):
    row = {'seq_name': scene_name}
    # Count the number of total samples/items per scene
    row['num_samples'] = get_total_weight(group[metric_cols[0]])
    for col in metric_cols:
        row[col] = get_weighted_average(group[col])
    scene_results.append(row)

scene_df = pd.DataFrame(scene_results)
display(scene_df)

,seq_name,num_samples,CD_16384_full,f_score_0.1_full,f_score_0.05_full,f_score_0.02_full
0,replica_pano_hotel_0_000,10,0.014564,0.996111,0.979585,0.822562
1,replica_pano_large_apartment_0_004,10,0.017431,0.994894,0.972379,0.760355
2,replica_pano_large_apartment_2_001,10,0.032704,0.950212,0.881032,0.438784
3,replica_pano_large_apartment_2_002,10,0.017383,0.994677,0.965446,0.744166
4,replica_pano_large_apartment_2_003,10,0.017779,0.994761,0.973290,0.717693
